In [2]:
# CELL 0 - LOAD DATA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Load Datasets
marketing_ads = pd.read_csv('marketing_ads_data_clean.csv')
sales_txn = pd.read_csv('sales_transaction_clean.csv')
ad_summary = pd.read_csv('ad_summary_clean.csv')

# Data Preview
print("=== Marketing Ads Data ===")
print(f"Shape: {marketing_ads.shape}")
print(f"Columns: {list(marketing_ads.columns)}")
print("\nPreview 5 baris pertama:")
print(marketing_ads.head(5).to_string())

print("\n=== Sales_txn ===")
print(f"Shape: {sales_txn.shape}")
print(f"Columns: {list(sales_txn.columns)}")
print("\nPreview 5 baris pertama:")
print(sales_txn.head(5).to_string())

print("\n=== ad_summary ===")
print(f"Shape: {ad_summary.shape}")
print(f"Columns: {list(ad_summary.columns)}")
print("\nPreview 5 baris pertama:")
print(ad_summary.head(5).to_string())

# Metrics Calculation
marketing_ads['CTR'] = marketing_ads['clicks'] / marketing_ads['impressions'] * 100
marketing_ads['CPC'] = marketing_ads['spend_idr'] / marketing_ads['clicks']

# Sanity Check
print(marketing_ads[['impressions', 'clicks', 'spend_idr', 'CTR', 'CPC']].head())

=== Marketing Ads Data ===
Shape: (1000, 12)
Columns: ['ad_id', 'campaign_name', 'audience_type', 'product_type', 'product_name', 'variant', 'date', 'impressions', 'clicks', 'spend_idr', 'sessions', 'add_to_cart']

Preview 5 baris pertama:
     ad_id         campaign_name audience_type product_type                   product_name         variant        date  impressions  clicks  spend_idr  sessions  add_to_cart
0  AD_0032    Awareness Campaign          cold        Serum    DermaGlow Brightening Serum    Video_Review  2023-01-06         8060     253   458896.0       216           29
1  AD_0110    Awareness Campaign          cold    Sunscreen  DermaGlow UV Shield Sunscreen    Video_Review  2023-09-01        17314     542   108070.0       452           72
2  AD_0137  Retargeting Campaign          warm    Sunscreen  DermaGlow UV Shield Sunscreen  Static_Catalog  2023-04-13        16374     180   135435.0       146            8
3  AD_0089  Retargeting Campaign          warm    Sunscreen  Der

In [3]:
# CELL 1 - CAMPAIGN PERFORMANCE OVERVIEW
# Metrics: CTR, CPC, Spend per Campaign Name
# Source: marketing_ads

campaign_performance = marketing_ads.groupby('campaign_name').agg(
    total_impressions=('impressions', 'sum'),
    total_clicks=('clicks', 'sum'),
    total_spend_idr=('spend_idr', 'sum'),
    total_sessions=('sessions', 'sum'),
    total_add_to_cart=('add_to_cart', 'sum'),
    avg_ctr=('CTR', 'mean'),
    avg_cpc=('CPC', 'mean')
).reset_index()

# Rekalkulasi CTR dan CPC dengan agregasi
campaign_performance['CTR_agg'] = campaign_performance['total_clicks'] / campaign_performance['total_impressions'] * 100
campaign_performance['CPC_agg'] = campaign_performance['total_spend_idr'] / campaign_performance['total_clicks']

# Session to Cart Rate per Campaign
campaign_performance['session_to_cart_rate'] = (campaign_performance['total_add_to_cart'] / campaign_performance['total_sessions']) * 100

cols = [
    'campaign_name', 'total_impressions', 'total_clicks', 'total_spend_idr',
    'CTR_agg', 'CPC_agg', 'total_sessions', 'total_add_to_cart', 'session_to_cart_rate'
]

print("=== Campaign Performance Overview ===")
print(campaign_performance[cols].to_string(index=False))

=== Campaign Performance Overview ===
       campaign_name  total_impressions  total_clicks  total_spend_idr  CTR_agg     CPC_agg  total_sessions  total_add_to_cart  session_to_cart_rate
  Awareness Campaign            4039964         74804      103530529.0 1.851601 1384.023969           63055               7792             12.357466
 Flash Sale Campaign            4134212         77443      105862007.0 1.873223 1366.966763           65962               7948             12.049362
Retargeting Campaign            4126286         75941       99963539.5 1.840420 1316.331619           64513               7783             12.064235


# CELL 1 - CAMPAIGN PERFORMANCE OVERVIEW (KEY FINDINGS)

## Key Findings:
- Ketiga Campaign selisih CTR tipis (1.84 - 1.87), Flash Sale Campaign sedikit unggul (1.87%), indikasi promo diskon    flash sale sangat menarik bagi audiens. 
- Retargeting Campaign paling murah CPC nya (Rp. 1,316 per klik), disusul flash sale (Rp. 1,366 per klik) kemudian Awareness Campaign paling mahal (Rp. 1,384 per klik). Indikasi audience warm di retargeting lebih mudah klik tanpa spend budget besar. 
- Awareness Campaign dengan Session to Cart Rate tertinggi (12.36%), selisih tipis dengan 2 campaign lain. 
- Skala spending ketiga campaign hampir sama (99 - 105 juta), perbandingan efisiensi jadi apple to apple. 

In [4]:
# CELL 2 - Spend Efficiency & ROAS per Campaign

# Filter Sales Transactions hanya status "SUccess"
sales_txn_success = sales_txn[sales_txn['order_status'] == 'Success']

# Join Marketing Ads untuk Campaign Name
sales_with_campaign = sales_txn_success.merge(
    marketing_ads[['ad_id', 'campaign_name', 'audience_type']].drop_duplicates(), 
    on= 'ad_id',
    how= 'left'
)

# Agregasi revenue per campaign
revenue_per_campaign = sales_with_campaign.groupby('campaign_name').agg(
    total_revenue_idr=('revenue_idr', 'sum'),
    total_transactions=('transaction_id', 'count')
).reset_index()

# Join dengan Campaign_Performance untuk dapat spend. 
campaign_roas = campaign_performance.merge(
    revenue_per_campaign,
    on='campaign_name',
    how='left'
)

# Hitung ROAS dan Revenue per Transactions
campaign_roas['ROAS'] = campaign_roas['total_revenue_idr'] / campaign_roas['total_spend_idr']
campaign_roas['avg_order_value'] = campaign_roas['total_revenue_idr'] / campaign_roas['total_transactions']

# Output
cols_roas = [
    'campaign_name', 'total_spend_idr', 'total_revenue_idr', 'total_transactions',
    'ROAS', 'avg_order_value'
]

print("=== Spend Efficiency & ROAS per Campaign ===")
print(campaign_roas[cols_roas].to_string(index=False))

=== Spend Efficiency & ROAS per Campaign ===
       campaign_name  total_spend_idr  total_revenue_idr  total_transactions     ROAS  avg_order_value
  Awareness Campaign      103530529.0        152801000.0                 546 1.475903    279855.311355
 Flash Sale Campaign      105862007.0        180347000.0                 641 1.703605    281352.574103
Retargeting Campaign       99963539.5        195734000.0                 674 1.958054    290406.528190


# CELL 2 — SPEND EFFICIENCY & ROAS PER CAMPAIGN

## Key Findings
- Retargeting Campaign mencatatkan ROAS tertinggi (1.96) dengan spend terendah (Rp 99,9 juta),
  menghasilkan revenue Rp 195,7 juta dari 674 transaksi.
- Flash Sale Campaign berada di posisi kedua (ROAS 1.70) dengan spend tertinggi (Rp 105,8 juta),
  mengindikasikan bahwa volume transaksi yang lebih besar tidak selalu berarti efisiensi lebih tinggi.
- Awareness Campaign mencatatkan ROAS terendah (1.48) — konsisten dengan perannya sebagai
  top-of-funnel campaign yang berfokus pada jangkauan, bukan konversi langsung.

## Insight
Dari sisi efisiensi spend, Retargeting Campaign adalah yang paling menguntungkan:
dengan budget paling kecil, ia menghasilkan revenue dan transaksi tertinggi.
Ini mengkonfirmasi bahwa warm audience — yang sudah familiar dengan produk —
membutuhkan lebih sedikit spend untuk berkonversi dibanding cold audience.

Flash Sale Campaign menarik volume transaksi besar, tetapi spend-nya juga tertinggi,
sehingga ROAS-nya kalah dari Retargeting. Ini menunjukkan bahwa diskon bisa mendorong
volume, tapi belum tentu efisiensi.

Awareness Campaign wajar memiliki ROAS terendah karena tujuannya bukan konversi langsung —
namun perlu diperhatikan apakah ia berkontribusi pada pipeline warm audience
yang kemudian dikonversi oleh Retargeting.

In [11]:
# CELL 3 - Audience Response per Campaign
# Metrics: CTR, CPC, Session to Cart Rate per Campaign x audience_type
# Source: marketing_ads

audience_response = marketing_ads.groupby(['campaign_name', 'audience_type']).agg(
    total_impressions=('impressions', 'sum'),
    total_clicks=('clicks', 'sum'),
    total_spend_idr=('spend_idr', 'sum'),
    total_sessions=('sessions', 'sum'),
    total_add_to_cart=('add_to_cart', 'sum')
).reset_index()


# Rekalkulasi metrik dengan agregasi
audience_response['CTR_agg'] = audience_response['total_clicks'] / audience_response['total_impressions'] * 100
audience_response['CPC_agg'] = audience_response['total_spend_idr'] / audience_response['total_clicks']
audience_response['session_to_cart_rate'] = audience_response['total_add_to_cart'] / audience_response['total_sessions'] * 100

# Tambah ROAS per audience_type
# Join Revenue dari sales_with_campaign
audience_revenue = sales_with_campaign.groupby(['campaign_name', 'audience_type']).agg(
    total_revenue_idr=('revenue_idr', 'sum'),
    total_transactions=('transaction_id', 'count')
).reset_index()

# Calculate ROAS
audience_response = audience_response.merge(audience_revenue, on=['campaign_name', 'audience_type'], how='left')
audience_response['ROAS'] = audience_response['total_revenue_idr'] / audience_response['total_spend_idr']

cols = [
    'campaign_name', 'audience_type',
    'total_impressions', 'CTR_agg', 'CPC_agg',
    'session_to_cart_rate', 'total_revenue_idr', 'ROAS'
]

print("=== Audience Response per Campaign ===")
print(audience_response[cols].to_string(index=False))

# Tambahan ringkasan per audience type
audience_summary = marketing_ads.groupby('audience_type').agg(
    total_impressions=('impressions', 'sum'),
    total_clicks=('clicks', 'sum'),
    total_spend_idr=('spend_idr', 'sum'),
    total_sessions=('sessions', 'sum'),
    total_add_to_cart=('add_to_cart', 'sum')
).reset_index()

audience_summary['CTR_agg'] = audience_summary['total_clicks'] / audience_summary['total_impressions'] * 100
audience_summary['CPC_agg'] = audience_summary['total_spend_idr'] / audience_summary['total_clicks']
audience_summary['session_to_cart_rate'] = audience_summary['total_add_to_cart'] / audience_summary['total_sessions'] * 100

print("\n=== Audience Summary (Cold vs Warm) ===")
print(audience_summary[['audience_type', 'CTR_agg', 'CPC_agg', 'session_to_cart_rate']].to_string(index=False))

=== Audience Response per Campaign ===
       campaign_name audience_type  total_impressions  CTR_agg     CPC_agg  session_to_cart_rate  total_revenue_idr     ROAS
  Awareness Campaign          cold            4039964 1.851601 1384.023969             12.357466        152801000.0 1.475903
 Flash Sale Campaign          cold            4134212 1.873223 1366.966763             12.049362        180347000.0 1.703605
Retargeting Campaign          warm            4126286 1.840420 1316.331619             12.064235        195734000.0 1.958054

=== Audience Summary (Cold vs Warm) ===
audience_type  CTR_agg     CPC_agg  session_to_cart_rate
         cold 1.862536 1375.347534             12.199943
         warm 1.840420 1316.331619             12.064235


# CELL 3 — AUDIENCE RESPONSE PER CAMPAIGN (KEY FINDINGS & INSIGHTS)

## Key Findings
- Retargeting Campaign (Warm Audience) mencatatkan ROAS tertinggi (1.96) dengan CPC
  termurah (Rp 1.316), menghasilkan total revenue Rp 195 juta.
- Flash Sale Campaign (Cold Audience) berada di posisi kedua (ROAS 1.70, CPC Rp 1.367),
  menghasilkan total revenue Rp 180 juta.
- Awareness Campaign (Cold Audience) mencatatkan ROAS terendah (1.48, CPC Rp 1.384),
  menghasilkan total revenue Rp 152 juta.
- Audience Summary (Cold vs Warm) menunjukkan CTR (1.86% vs 1.84%), CPC (Rp 1.375 vs Rp 1.316),
  dan session to cart rate (12.2% vs 12.1%), selisihnya tipis di semua metrik engagement.

## Catatan Penting — Keterbatasan Data
Setiap campaign type di dataset ini hanya terasosiasi dengan satu audience type
(Awareness & Flash Sale - cold, Retargeting - warm). Karena itu, perbandingan
"cold vs warm" pada dasarnya setara dengan perbandingan "Awareness + Flash Sale vs Retargeting".
Tidak bisa disimpulkan murni efek audience type, karena variabel campaign dan audience
bergerak bersamaan (confounded).

## Insight
Selisih ROAS dan Revenue antar campaign jauh lebih besar dibanding selisih CTR,
CPC, atau session to cart rate yang semuanya relatif seragam di kisaran 1.8% CTR
dan 12% session to cart. Mengindikasikan perbedaan performa bisnis
(ROAS, Revenue) lebih didorong oleh efisiensi biaya per klik (CPC) ketimbang
seberapa menarik iklan tersebut (CTR) atau seberapa niat audiens untuk checkout
(session to cart rate).

Karena CPC warm audience secara konsisten lebih murah dibanding cold, dan engagement
rate nya setara, warm audience (Retargeting) tetap menjadi segmen paling efisien
secara biaya namun catatan limitasi di atas perlu disertakan agar kesimpulan ini
tidak dibaca sebagai audience type yang murni unggul terlepas dari campaign type nya.

In [6]:
# CELL 4 - PRODUCT PERFROMANCE per CAMPAIGN
# Matrics: Revenue, Transactions, ROAS per Campaign x product_name
# Source: marketing_ads

sales_with_campaign = sales_txn_success.merge(
    marketing_ads[['ad_id', 'campaign_name', 'audience_type', 'product_name']].drop_duplicates(),
    on='ad_id',
    how='left'
).drop(columns=['product_name_x']).rename(columns={'product_name_y': 'product_name'})

# Langkah 1: Spend per Campaign x product_name
product_spend = marketing_ads.groupby(['campaign_name', 'product_name']).agg(
    total_impressions=('impressions', 'sum'),
    total_clicks=('clicks', 'sum'),
    total_spend_idr=('spend_idr', 'sum')
).reset_index()

# Langkah 2: Revenue per Campaign x product_name
product_revenue = sales_with_campaign.groupby(['campaign_name', 'product_name']).agg(
    total_revenue_idr=('revenue_idr', 'sum'),
    total_transaction=('transaction_id', 'count')
).reset_index()

# Langkah 3: Join Spend dan Revenue. 
product_performance = product_spend.merge(
    product_revenue, on=['campaign_name', 'product_name'], how='left'
)

# Langkah 4: Hitung ROAS per Product per Campaign
product_performance['ROAS'] = product_performance['total_revenue_idr'] / product_performance['total_spend_idr']

# Langkah 5: List top ptoduct per campaign
product_performance_sorted = product_performance.sort_values(
    ['campaign_name', 'total_revenue_idr'], ascending=[True, False]
)

cols = [
    'campaign_name', 'product_name', 'total_spend_idr',
    'total_revenue_idr', 'total_transaction', 'ROAS'
]

print("=== Product Performance per Campaign")
print(product_performance_sorted[cols].to_string(index=False))

=== Product Performance per Campaign
       campaign_name                    product_name  total_spend_idr  total_revenue_idr  total_transaction     ROAS
  Awareness Campaign     DermaGlow Brightening Serum       26433165.0         43586000.0                151 1.648913
  Awareness Campaign   DermaGlow UV Shield Sunscreen       26571522.0         42454000.0                149 1.597726
  Awareness Campaign DermaGlow Hydrating Moisturizer       30760651.0         41401000.0                152 1.345908
  Awareness Campaign       DermaGlow Gentle Cleanser       19765191.0         25360000.0                 94 1.283064
 Flash Sale Campaign DermaGlow Hydrating Moisturizer       30862647.0         49242000.0                176 1.595521
 Flash Sale Campaign   DermaGlow UV Shield Sunscreen       23565944.0         48121000.0                174 2.041972
 Flash Sale Campaign     DermaGlow Brightening Serum       27577304.0         42502000.0                149 1.541195
 Flash Sale Campaign       

# CELL 4 — PRODUCT PERFORMANCE PER CAMPAIGN (Key Findings & Insight)

## Key Findings
- Awareness Campaign: Brightening Serum mencatatkan ROAS tertinggi (1.65) dengan revenue
  Rp 43,6 juta (151 transaksi), diikuti UV Shield Sunscreen (ROAS 1.60) dan Moisturizer (1.35).
  Gentle Cleanser menjadi yang terendah (ROAS 1.28).
- Flash Sale Campaign: UV Shield Sunscreen mencatatkan ROAS tertinggi (2.04) dengan revenue
  Rp 48,1 juta (174 transaksi), meski spendnya bukan yang terbesar. Moisturizer unggul di
  volume transaksi (176) tapi ROAS-nya lebih rendah (1.60).
- Retargeting Campaign: Gentle Cleanser mendominasi dengan ROAS 2.86 dan revenue tertinggi
  Rp 60,3 juta (205 transaksi) — angka ROAS tertinggi di seluruh tabel.

## Insight
- Gentle Cleanser menunjukkan pola yang paling mencolok: ROAS rendah di Awareness (1.28)
  dan Flash Sale (1.70), tapi meledak di Retargeting (2.86). Produk ini sangat responsif
  terhadap warm audience dan paling efisien jika di-retarget.
- UV Shield Sunscreen konsisten perform di semua campaign (ROAS 1.60 → 2.04 → 1.87),
  menjadikannya produk paling stabil dan aman untuk dialokasikan di campaign manapun.
- Moisturizer menunjukkan pola terbalik dari Gentle Cleanser — ROAS tertinggi di Flash Sale
  (1.60) tapi terendah di Retargeting (1.43), mengindikasikan produk ini lebih cocok
  didorong lewat promo diskon dibanding retargeting.
- Implikasi budget: Gentle Cleanser sebaiknya diprioritaskan di Retargeting,
  UV Shield Sunscreen bisa dijalankan di semua campaign, dan Moisturizer
  paling efisien jika difokuskan ke Flash Sale.

In [7]:
# Cell 5 — Export CSV untuk Power BI

# Tabel yang di export:
# 1. campaign_performance   — Cell 1: CTR, CPC, Spend per campaign
# 2. campaign_roas          — Cell 2: ROAS, Revenue per campaign
# 3. audience_response      — Cell 3: Metrics per campaign x audience
# 4. audience_summary       — Cell 3: Cold vs Warm summary
# 5. product_performance    — Cell 4: Product performance per campaign

campaign_performance.to_csv('campaign_performance.csv', index=False)
campaign_roas.to_csv('campaign_roas.csv', index=False)
audience_response.to_csv('audience_response.csv', index=False)
audience_summary.to_csv('audience_summary.csv', index=False)
product_performance_sorted.to_csv('product_performance.csv', index=False)

print("=== Export Selesai ===")
print("File di export:")
print("  - campaign_performance.csv")
print("  - campaign_roas.csv")
print("  - audience_response.csv")
print("  - audience_summary.csv")
print("  - product_performance.csv")

=== Export Selesai ===
File di export:
  - campaign_performance.csv
  - campaign_roas.csv
  - audience_response.csv
  - audience_summary.csv
  - product_performance.csv


In [13]:
# CELL 6 - Re Export Audience Response
audience_response.to_csv('audience_response.csv', index=False)
print("Export Audience Response Berhasil")

Export Audience Response Berhasil
